<a href="https://colab.research.google.com/github/abirbhab-dasgupta/word-prediction-lstm/blob/main/LSTM_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")


In [5]:
df = pd.read_csv('qoute_dataset.csv')

In [6]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [7]:
df.shape

(3038, 2)

In [8]:
quotes= df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [9]:
quotes = quotes.str.lower()

In [10]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [11]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [13]:
vocab_size = 10000
tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quotes)

In [14]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [15]:
sequence = tokenizer.texts_to_sequences(quotes)

In [16]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [17]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [18]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [19]:
print(len(X))
print(len(y))

85271
85271


In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [21]:
X_padded = pad_sequences(X,maxlen= max(len(x) for x in X),padding='pre')

In [22]:
X_padded.shape

(85271, 745)

In [23]:
y = np.array(y)

In [24]:
y.shape

(85271,)

In [25]:
from tensorflow.keras.utils import to_categorical

In [26]:
y_one_hot = to_categorical(y, num_classes=vocab_size)
display(y_one_hot.shape)

(85271, 10000)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN, LSTM, Dense

In [28]:
embedding_dim = 50
rnn_units = 128

In [29]:
rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max(len(x) for x in X)))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max(len(x) for x in X)))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
epochs = 10
batch_size = 128

In [32]:
history_rnn = rnn_model.fit(X_padded, y_one_hot, epochs=epochs, batch_size=batch_size,validation_split=0.2)

Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 46s 76ms/step - accuracy: 0.0438 - loss: 6.7163 - val_accuracy: 0.0531 - val_loss: 6.6567
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.0702 - loss: 6.1601 - val_accuracy: 0.0841 - val_loss: 6.4862
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.0957 - loss: 5.8156 - val_accuracy: 0.0939 - val_loss: 6.4031
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.1120 - loss: 5.5162 - val_accuracy: 0.1027 - val_loss: 6.4251
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.1254 - loss: 5.2501 - val_accuracy: 0.1072 - val_loss: 6.4637
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.1382 - loss: 5.0132 - val_accuracy: 0.1067 - val_loss: 6.5314
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.1504 - loss: 4.7954 - val_accuracy: 0.1078 - val_loss: 6.5889
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 41s 67ms/step - accuracy: 0.1648 - loss: 4.5916 - 

In [34]:
rnn_model.save('rnn_model.h5')

In [35]:
history_lstm = lstm_model.fit(X_padded, y_one_hot, epochs=100, batch_size=128,validation_split=0.2)

Epoch 1/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 38s 62ms/step - accuracy: 0.0386 - loss: 6.7614 - val_accuracy: 0.0469 - val_loss: 6.7385
Epoch 2/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 55ms/step - accuracy: 0.0548 - loss: 6.3350 - val_accuracy: 0.0632 - val_loss: 6.6647
Epoch 3/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - accuracy: 0.0736 - loss: 6.0897 - val_accuracy: 0.0822 - val_loss: 6.5584
Epoch 4/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 0.0925 - loss: 5.8785 - val_accuracy: 0.0905 - val_loss: 6.5249
Epoch 5/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 58ms/step - accuracy: 0.1039 - loss: 5.7061 - val_accuracy: 0.0960 - val_loss: 6.4973
Epoch 6/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 57ms/step - accuracy: 0.1145 - loss: 5.5402 - val_accuracy: 0.0987 - val_loss: 6.4980
Epoch 7/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.1232 - loss: 5.3811 - val_accuracy: 0.1043 - val_loss: 6.5331
Epoch 8/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 58ms/step - accuracy: 0.1312 - loss: 5

In [36]:
lstm_model.save('lstm_model.h5')

In [41]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [42]:
max_len = max(len(x) for x in X)

In [43]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [48]:
seed_text = "She is"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

the


In [53]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [54]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generate_text)

are you a  brain life turns it into the light and the past


In [57]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [58]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)